In [ ]:
import sys
# TO CHANGE
BASEDIR = "../../../../.."
sys.path.insert(0, BASEDIR)

In [ ]:
from pprint import pprint
import gc

In [ ]:
from src.db_drivers.vector_driver import EmbedderModel, EmbedderModelConfig
from src.kg_model.nodestree_model import NodesTreeModel, NodesTreeModelConfig
from src.agents import AgentDriver, AgentDriverConfig
from src.agents.configs import DEFAULT_AGENT_CONFIGS
from src.utils import TripletCreator, NodeCreator, RelationCreator

## Пример работы с NodesTreeModel-частью модели графа знаний

1. Инициализация класса, выполняющего функции embedder-модели

In [ ]:
EMBEDDER_MODEL_PATH = '../../../../../models/intfloat/multilingual-e5-base' # PATH TO APPROPRIATE EMBEDDER-MODEL
embedder_config = EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH)
pprint(embedder_config)

In [ ]:
embedder = EmbedderModel(embedder_config)

In [ ]:
embedders_mapping = {
    'leaf_dense_nodes': embedder,
    'summ_dense_nodes': embedder
}

2. Инициализация Connector-класса к LLM-моделе

In [ ]:
adriver_config = AgentDriverConfig(name='ollama', agent_config=DEFAULT_AGENT_CONFIGS['ollama'])
pprint(adriver_config, width=200)
agent = AgentDriver.select(adriver_config)
agent.open_connection()

In [ ]:
output = agent.generate(system_prompt="You are helpful assistant.", user_prompt="What is wrong with human kind?")
print(output[1])
print(output[0])

3. Инициализация NodesTreeModel-модели

In [ ]:
nodestree_config = NodesTreeModelConfig(lang='ru')
pprint(nodestree_config, compact=True)

In [ ]:
nodestree_model = NodesTreeModel(agent, embedders_mapping, nodestree_config)

In [ ]:
nodestree_model.clear()

In [ ]:
nodestree_model.count_items(detailed=True)

In [ ]:
print("Консистентность модели дерева вершин:", nodestree_model.check_consistency())

4. Добавление новых вершин в модель дерева вершин

In [ ]:
TRIPLET_EXAMPLE1 = TripletCreator.create(
    start_node=NodeCreator.create(n_type='object', name='саша'), 
    relation=RelationCreator.create(r_type='simple', name='шла'), 
    end_node=NodeCreator.create(n_type='object', name='улица'))
TRIPLET_EXAMPLE2 = TripletCreator.create(
    start_node=NodeCreator.create(n_type='object', name='маша'), 
    relation=RelationCreator.create(r_type='simple', name='шла'), 
    end_node=NodeCreator.create(n_type='object', name='шоссе'))
TRIPLET_EXAMPLE3 = TripletCreator.create(
    start_node=NodeCreator.create(n_type='object', name='теплоход'), 
    relation=RelationCreator.create(r_type='simple', name='плыл'), 
    end_node=NodeCreator.create(n_type='object', name='канал'))
TRIPLET_EXAMPLE4 = TripletCreator.create(
    start_node=NodeCreator.create(n_type='object', name='моторная лодка'), 
    relation=RelationCreator.create(r_type='simple', name='плыл'), 
    end_node=NodeCreator.create(n_type='object', name='река'))

In [ ]:
pprint(TRIPLET_EXAMPLE1, width=200)
pprint(TRIPLET_EXAMPLE2, width=200)
pprint(TRIPLET_EXAMPLE3, width=200)
pprint(TRIPLET_EXAMPLE4, width=200)

In [ ]:
nodestree_model.expand_tree([TRIPLET_EXAMPLE1, TRIPLET_EXAMPLE2, TRIPLET_EXAMPLE3, TRIPLET_EXAMPLE4])
print("Количество объектов в древовидной моделе хранения вершин: ")
pprint(nodestree_model.count_items(detailed=True))
print("Консистентность модели дерева вершин:", nodestree_model.check_consistency())

5. Сопоставление сущности с вершинами из соответствующей древовидной модели

In [ ]:
ENTITE_EXAMPLE1 = "олеся"
ENTITE_EXAMPLE2 = "крейсер"
ENTITE_EXAMPLE3 = "проезжая чать"

In [ ]:
matched_nodes1 = nodestree_model.match_entity2objects(ENTITE_EXAMPLE1)
print("query_entitie: ", ENTITE_EXAMPLE1)
print("matched nodes: ", list(map(lambda inst: inst.text, matched_nodes1)))

matched_nodes2 = nodestree_model.match_entity2objects(ENTITE_EXAMPLE2)
print("query_entitie: ", ENTITE_EXAMPLE2)
print("matched nodes: ", list(map(lambda inst: inst.text, matched_nodes2)))

matched_nodes3 = nodestree_model.match_entity2objects(ENTITE_EXAMPLE3)
print("query_entitie: ", ENTITE_EXAMPLE3)
print("matched nodes: ", list(map(lambda inst: inst.text, matched_nodes3)))

nodestree_model.count_items()
nodestree_model.check_consistency()

6. Полное удаление содержимого модели дерева вершин

In [ ]:
nodestree_model.clear()
nodestree_model.count_items(detailed=True)

In [ ]:
nodestree_model.close_connections()
del nodestree_model 
gc.collect()